In [ ]:
import pandas as pd
import networkx as nx
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score
import matplotlib.pyplot as plt
import matplotlib.cm as cm

import ast



In [ ]:
#df = pd.read_csv("")

#pred_file = "data/output/pscan_output/labels/lfr_5000.labels.tsv"
# Load predictions and ground truth data

# pred_df = pd.read_csv(pred_file, sep="\t").rename(
#     columns={"label": "predicted_label"}
# )



#gt_adjlist_file = "data/adjlists/lfr_5000.adjlist"
#pred_adjlist_file = "data/output/pscan_output/labels/lfr_5000.labels.tsv"

gt_file = "similarity/data/labels/lfr_500.labels.tsv"
gt_df = pd.read_csv(gt_file, sep="\t").rename(
    columns={"label": "gt_label"}
)

pred_file = "similarity/cluster_code_alliancecan/data/output/clusters/lfr_500.csv"
pred_df = pd.read_csv(pred_file, header=None, names=["predicted_label", "node"])
pred_df["node"] = pred_df["node"].apply(ast.literal_eval)
pred_df = pred_df.explode("node").reset_index(drop=True)
pred_df["node"] = pred_df["node"].astype(int)


df = pd.merge(
    gt_df,
    pred_df,
    on="node",
).reset_index()
print(df.head())

In [ ]:
df

In [ ]:
pred_df

In [ ]:
gt_df

In [ ]:




# compute ARI
ari = adjusted_rand_score(df["gt_label"], df["predicted_label"])
print(f"adjusted rand index score: {ari}")

# compute NMI
nmi = normalized_mutual_info_score(df["gt_label"], df["predicted_label"])
print(f"normalized mutual info score: {nmi}")


In [ ]:
G = nx.read_adjlist(gt_adjlist_file)



# generate colour map
classes = df["predicted_label"].unique()
cmap = cm.get_cmap("tab10", len(classes))
class_colors = {cls: cmap(i) for i, cls in enumerate(classes)}
node_colors = [
    class_colors[df.loc[df['node'] == int(n)]["gt_label"].values[0]]
    for n in G.nodes()
]
# for n in G.nodes():
#     G.nodes[n]["class"] = node_lookup.get(n, "unknown")

# generate graph pos
pos = nx.spectral_layout(G)
#pos = nx.spring_layout(G, pos=pos, iterations=20, seed=42)
#pos = nx.nx_agraph.graphviz_layout(G, prog="sfdp")


In [ ]:
plt.figure(figsize=(16, 9))
nx.draw(G, 
        pos, 
        with_labels=False,
        node_color=node_colors,
        node_size=30,
        edge_color="grey",
        )
plt.title("Graph from .adjlist")
plt.tight_layout()
plt.show()